# Train LineVul on `cwe_mix`

No Joern, no DGL, no GNN. Enable GPU: **Runtime → Change runtime type → T4 GPU**.

This notebook trains **one** new checkpoint: `linevul_cwe_mix`.

- Data: `data/cwe_mix/` (all of CWE-synth **plus** filtered real C pair-diffs). Already built — do not regenerate Big-Vul.
- Output: Drive `FYP_training_output/linevul_cwe_mix`
- Do **not** overwrite `linevul`, `linevul_rank`, or `linevul_cwe`.

Run cells **in order: 1 → 2 → 3 → 4**. Watch `val_line_mcc`.


## 1. Mount Drive and locate the project


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
from pathlib import Path

MYDRIVE = '/content/drive/MyDrive'

def find_root():
    for c in [
        f'{MYDRIVE}/FYP/linevul',
        f'{MYDRIVE}/linevul',
        '/content/linevul',
    ]:
        if os.path.isdir(c) and os.path.isdir(os.path.join(c, 'linevul')):
            return c
    zips = glob.glob(f'{MYDRIVE}/**/linevul.zip', recursive=True)
    if zips:
        !unzip -qo {zips[0]} -d /content
        if os.path.isdir('/content/linevul'):
            return '/content/linevul'
    raise FileNotFoundError('Upload linevul/ or linevul.zip to My Drive/FYP/')

ROOT = find_root()
os.chdir(ROOT)
print('ROOT', ROOT)
mix = Path('data/cwe_mix/train.jsonl')
print('cwe_mix train exists:', mix.is_file(), mix)


## 2. Install (plain transformers — no DGL)


In [ ]:
!pip -q install -r requirements.txt
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit('Enable a T4 GPU before training.')


## 3. Confirm `cwe_mix` (do not rebuild Big-Vul)

Expected: train **11312**, val **1412**, test **1412**.
That is 7040/880/880 synth plus the filtered C pairs.
If this cell fails, the zip is missing `data/cwe_mix/` — re-upload the project, do **not** run `prepare_subset.py`.


In [ ]:
from pathlib import Path
import json

stats_path = Path('data/cwe_mix/stats.json')
train_path = Path('data/cwe_mix/train.jsonl')
val_path = Path('data/cwe_mix/val.jsonl')
if not stats_path.is_file() or not train_path.is_file() or not val_path.is_file():
    raise FileNotFoundError(
        'data/cwe_mix/ is missing. Upload a linevul.zip that includes data/cwe_mix/. '
        'Do not run scripts/prepare_subset.py — that is the old Big-Vul set.'
    )
stats = json.loads(stats_path.read_text())
print(json.dumps(stats, indent=2))
assert stats.get('kind') == 'cwe_synth_plus_filtered_c_pairs', stats.get('kind')
assert stats.get('train') == 11312, stats.get('train')
assert stats.get('val') == 1412, stats.get('val')
print('ok — training will use data/cwe_mix')


## 4. Train `linevul_cwe_mix` from scratch

Writes **only** `FYP_training_output/linevul_cwe_mix`.
Leaves `linevul_cwe` on Drive alone so you can compare gates later.


In [ ]:
from pathlib import Path

OUT = '/content/drive/MyDrive/FYP_training_output/linevul_cwe_mix'
!python -m linevul.train \
  --data-dir data/cwe_mix \
  --out-dir {OUT} \
  --epochs 6 \
  --batch-size 8 \
  --lr 2e-5 \
  --stmtweight auto \
  --patience 3 \
  --ranking-lambda 0.5 \
  --ranking-margin 1.0
print('checkpoint dir', OUT)
print(list(Path(OUT).glob('*')))


## 5. After training — on your Mac, not in Colab

Download Drive `FYP_training_output/linevul_cwe_mix` into:

`~/Desktop/projects/FYP/linevul/checkpoints/linevul_cwe_mix/`

Keep `checkpoints/linevul_cwe` as the current demo until the gate is better.

```
cd ~/Desktop/projects/FYP/linevul
LINEVUL_CHECKPOINT=checkpoints/linevul_cwe_mix python scripts/macos_eval.py
```

Switch the app to `linevul_cwe_mix` only if it beats **linevul_cwe** on the CWE gate: pairwise AUC **> 0.84** and top-1 **> 7/10**. If not, keep `linevul_cwe`.

Old cells (Big-Vul `data/subset`, `linevul`, `linevul_rank`, synth-only `linevul_cwe`) were removed from this notebook on purpose. Those checkpoints stay on disk as baselines.
